In [15]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [16]:
### Fourier Domain Adaptation

@torch.no_grad()
def fda(src_img: torch.Tensor, tgt_img: torch.Tensor, beta) -> torch.Tensor:
    C, H, W = src_img.shape

    src_fft = torch.fft.fft2(src_img, dim=(-2, -1))
    tgt_fft = torch.fft.fft2(tgt_img, dim=(-2, -1))

    src_amp, src_phase = torch.abs(src_fft), torch.angle(src_fft)
    tgt_amp = torch.abs(tgt_fft)

    src_amp_shift = torch.fft.fftshift(src_amp, dim=(-2, -1))
    tgt_amp_shift = torch.fft.fftshift(tgt_amp, dim=(-2, -1))

    b_h = int(beta * H)
    b_w = int(beta * W)
    c_h, c_w = H // 2, W // 2
    h1, h2 = c_h - b_h, c_h + b_h
    w1, w2 = c_w - b_w, c_w + b_w

    src_amp_shift[..., h1:h2, w1:w2] = tgt_amp_shift[..., h1:h2, w1:w2]

    src_amp_new = torch.fft.ifftshift(src_amp_shift, dim=(-2, -1))
    src_fft_new = src_amp_new * torch.exp(1j * src_phase)
    src_in_tgt = torch.fft.ifft2(src_fft_new, dim=(-2, -1)).real

    return torch.clamp(src_in_tgt, 0.0, 255.0)

In [17]:
### Removing Cloud Cover

import cv2
import numpy as np

def white_fraction_hsv(img_bgr: np.ndarray, v_thresh: int = 230, s_thresh: int = 40, border_ignore: float = 0.05):
    h, w = img_bgr.shape[:2]
    if border_ignore > 0:
        bh = int(h * border_ignore)
        bw = int(w * border_ignore)
        img_bgr = img_bgr[bh:h-bh, bw:w-bw]

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2HSV)
    H, S, V = hsv[...,0], hsv[...,1], hsv[...,2]
    white_like = (V >= v_thresh) & (S <= s_thresh)
    return white_like.mean()


In [18]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

normalize_05 = transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
to_tensor = transforms.ToTensor()
imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cloud_thresh = 0.15
max_tries = 10
removeClouds = False

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (to_tensor(mask) > 0).float()

        return img, mask
    
class SourceDataset(Dataset):
    def __init__(self, source_ds, target_ds, beta):
        self.source = source_ds
        self.target = target_ds
        self.beta = beta
    
    def __len__(self):
        return len(self.source)
    
    def sample_target_img(self):
        random.seed()
        if(removeClouds):
            for _ in range(max_tries):
                tgt_path = random.choice(self.target)
                img = tif.imread(tgt_path)

                white_frac = white_fraction_hsv(img)
                if white_frac <= cloud_thresh:
                    return to_tensor(img)
            
            return to_tensor(img)
        else:
            tgt_path = random.choice(self.target)
            img = tif.imread(tgt_path)
            return to_tensor(img)
    
    def __getitem__(self, idx):
        src = self.source[idx]
        mask = src.replace("img", "mask")

        src_img = to_tensor(tif.imread(src)).float().to(device, non_blocking=True)
        src_mask = (to_tensor(tif.imread(mask)) > 0).float().to(device, non_blocking=True)

        tgt_img = self.sample_target_img().float().to(device, non_blocking=True)

        src_img = (fda(src_img*255, tgt_img*255, beta=self.beta)/255.0).clamp(0,1)
        src_img = normalize_05(src_img)

        return src_img, src_mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

In [19]:
def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))


### COMPUTE R (NEGATIVE PIXELS / POSITIVE PIXELS)
r = 4.5
pos_weight = torch.tensor([r]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [20]:
import torch.nn.functional as F

def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name, threshold=0.6):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            bce  = criterion(logits, masks)
            dice = dice_loss(logits, masks)
            loss = bce + dice

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                bce  = criterion(logits, masks)
                dice = dice_loss(logits, masks)
                loss = bce + dice
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader, threshold=0.6):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold
                    
            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    return getMetrics(TP, TN, FP, FN)

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

beta = 0.15
beta_name = "15"

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, beta)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/new_fda/{beta_name}/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/new_fda/{beta_name}.csv", "w") as f:
    f.write(output)


In [14]:
# Beta sweep

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

beta_values = [0.01, 0.03, 0.05, 0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.15]

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for beta in beta_values:
    for source_region in regions_dict:
        source_images = regions_dict[source_region]
        source_masks = [f.replace("img", "mask") for f in source_images]

        train_img, val_img, train_mask, val_mask = train_test_split(
            source_images, source_masks, test_size=.2, random_state=42
        )

        target_region = "Longxi River (UAV)"

        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, beta)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/fda_sweep/{str(beta)[-2:]}_{source_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{beta}, {source_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/new_fda/fda_sweep.csv", "w") as f:
    f.write(output)

region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.793, 1.780, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 2, 1.761, 1.763, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 3, 1.729, 1.729, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 4, 1.671, 1.644, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 5, 1.613, 1.627, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 6, 1.476, 4.526, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 7, 1.401, 6.967, 0.0000, 0.0000, 0.0000, 0.0000, 0.4685, 0.9369
Hokkaido Iburi-Tobu, 8, 1.307, 1.924, 0.5556, 0.0431, 0.0801, 0.0417, 0.4896, 0.9377
Hokkaido Iburi-Tobu, 9, 1.240, 1.164, 0.6609, 0.3021, 0.4147, 0.2616, 0.6035, 0.9464
Hokkaido Iburi-Tobu, 10, 1.126, 1.094, 0.7327, 0.4059, 0.5224, 0.3536, 0.6529, 0.9534
Hokkaido Iburi-Tobu, 11, 1.094, 1.025, 0.6702, 0.5454, 0.6014, 0.4300, 0.69

In [23]:
### VISUALIZER
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as TF
import pandas as pd

@torch.no_grad()
def visualize_predictions_with_metrics(
    model_base_path,
    model_fda_path,
    target_region="Longxi River (UAV)",
    threshold=0.6
):
    os.makedirs("../../results/longxi_case/visuals", exist_ok=True)
    out_csv = "../../results/longxi_case/visuals/visual_metrics.csv"

    print(f"\n[Visualizing predictions for {target_region}]")

    model_base = ResNetUNet().to(device)
    model_base.load_state_dict(torch.load(model_base_path, map_location=device))
    model_base.eval()

    model_fda = ResNetUNet().to(device)
    model_fda.load_state_dict(torch.load(model_fda_path, map_location=device))
    model_fda.eval()

    target_imgs = regions_dict[target_region]
    
    results = []

    for idx, img_path in enumerate(target_imgs):
        img_name = os.path.basename(img_path)
        img = tif.imread(img_path)
        mask = tif.imread(img_path.replace("img", "mask"))
        img_t = imageTransform(img).unsqueeze(0).to(device)
        mask_t = (to_tensor(mask) > 0).float().to(device)

        def evaluate(pred_logits):
            if pred_logits.shape[-2:] != mask_t.shape[-2:]:
                pred_logits = F.interpolate(pred_logits, size=mask_t.shape[-2:], mode="bilinear", align_corners=False)
            preds = (torch.sigmoid(pred_logits) > threshold).float()
            TP = ((preds == 1) & (mask_t == 1)).sum().item()
            FP = ((preds == 1) & (mask_t == 0)).sum().item()
            FN = ((preds == 0) & (mask_t == 1)).sum().item()
            TN = ((preds == 0) & (mask_t == 0)).sum().item()
            precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
            return preds.cpu().squeeze(0), precision, recall, f1, iou

        out_b = model_base(img_t)
        pred_b, p_b, r_b, f1_b, iou_b = evaluate(out_b)

        out_f = model_fda(img_t)
        pred_f, p_f, r_f, f1_f, iou_f = evaluate(out_f)

        rgb = TF.normalize(to_tensor(img), [0,0,0], [1,1,1])
        gt = mask_t.cpu().squeeze(0)
        comparison = make_grid([rgb, gt.expand_as(rgb), pred_b.expand_as(rgb), pred_f.expand_as(rgb)], nrow=4)
        out_path = f"../../results/longxi_case/visuals/{idx:02d}_{img_name}_cmp.png"
        save_image(comparison, out_path)

        results.append({
            "image": img_name,
            "iou_baseline": iou_b,
            "iou_fda": iou_f,
            "precision_baseline": p_b,
            "precision_fda": p_f,
            "recall_baseline": r_b,
            "recall_fda": r_f,
            "f1_baseline": f1_b,
            "f1_fda": f1_f,
            "path": out_path
        })
        print(f"Saved {out_path} | IoU baseline={iou_b:.3f}, FDA={iou_f:.3f}")

    pd.DataFrame(results).to_csv(out_csv, index=False)
    print(f"\n[Done] Saved metrics for {len(results)} images → {out_csv}")

In [ ]:
baseline_model_path = "../../models/baseline/source_only_100/Moxitaidi (UAV-0.6m).pth"
fda_model_path = "../../models/new_fda/09/Moxitaidi (UAV-0.6m)_Longxi River (UAV).pth"

visualize_predictions_with_metrics(
    baseline_model_path,
    fda_model_path,
)


[Visualizing predictions for Longxi River (UAV)]
Saved ../../results/longxi_case/visuals/00_longxihe_UAV0166.tif_cmp.png | IoU baseline=0.174, FDA=0.444
Saved ../../results/longxi_case/visuals/01_longxihe_UAV1966.tif_cmp.png | IoU baseline=0.000, FDA=0.802
Saved ../../results/longxi_case/visuals/02_longxihe_UAV0941.tif_cmp.png | IoU baseline=0.047, FDA=0.414
Saved ../../results/longxi_case/visuals/03_longxihe_UAV0844.tif_cmp.png | IoU baseline=0.002, FDA=0.309
Saved ../../results/longxi_case/visuals/04_longxihe_UAV1924.tif_cmp.png | IoU baseline=0.000, FDA=0.371
Saved ../../results/longxi_case/visuals/05_longxihe_UAV1466.tif_cmp.png | IoU baseline=0.000, FDA=0.263
Saved ../../results/longxi_case/visuals/06_longxihe_UAV1284.tif_cmp.png | IoU baseline=0.000, FDA=0.286
Saved ../../results/longxi_case/visuals/07_longxihe_UAV0962.tif_cmp.png | IoU baseline=0.000, FDA=0.272
Saved ../../results/longxi_case/visuals/08_longxihe_UAV0943.tif_cmp.png | IoU baseline=0.135, FDA=0.508
Saved ../../re

In [ ]:
### BOUNDARY F1

import cv2
import numpy as np
import torch

def boundary_f1_score(pred, gt, dilation_ratio=0.02):
    if pred.ndim == 3:
        pred = pred.squeeze(0)
    if gt.ndim == 3:
        gt = gt.squeeze(0)

    pred_np = pred.detach().cpu().numpy().astype(np.uint8)
    gt_np   = gt.detach().cpu().numpy().astype(np.uint8)

    pred_edge = cv2.Canny(pred_np*255, 100, 200)
    gt_edge   = cv2.Canny(gt_np*255, 100, 200)

    diag = np.sqrt(pred_np.shape[0]**2 + pred_np.shape[1]**2)
    tol  = max(1, int(dilation_ratio * diag))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tol,tol))

    pred_dil = cv2.dilate(pred_edge, kernel)
    gt_dil   = cv2.dilate(gt_edge, kernel)

    tp = np.logical_and(pred_edge>0, gt_dil>0).sum()
    fp = np.logical_and(pred_edge>0, gt_dil==0).sum()
    fn = np.logical_and(gt_edge>0, pred_dil==0).sum()

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    bf1       = 2 * precision * recall / (precision + recall + 1e-8)
    return bf1

In [ ]:
### SPECTRAL ANALYSIS

import numpy as np, tifffile as tif, os, random
from tqdm import tqdm
from scipy.fft import fft2, fftshift

def compute_spectral_features(img):
    """Return (centroid, LF/HF ratio) for one image."""
    if img.ndim == 3:
        img = np.mean(img, axis=2)
    H, W = img.shape
    F = np.abs(fftshift(fft2(img)))
    F /= F.sum()

    y, x = np.indices((H, W))
    r = np.sqrt((x - W/2)**2 + (y - H/2)**2)
    centroid = (F * r).sum() / F.sum()

    r_norm = r / r.max()
    lf_energy = F[r_norm < 0.25].sum()
    hf_energy = F[r_norm >= 0.25].sum()
    lf_hf = lf_energy / (hf_energy + 1e-8)
    return centroid, lf_hf

def spectral_csv_for_region(region_key, out_csv, max_samples=100):
    """Compute spectral stats for all images in a region."""
    imgs = regions_dict[region_key]
    imgs = random.sample(imgs, min(max_samples, len(imgs)))
    rows = []
    for p in tqdm(imgs, desc=f"Spectral {region_key}"):
        img = tif.imread(p)
        c, r = compute_spectral_features(img)
        rows.append({"image": os.path.basename(p), "centroid": c, "lf_hf": r})
    import pandas as pd
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"Saved → {out_csv}")
    return df

def summarize_spectral_csv(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    return {
        "mean_centroid": df["centroid"].mean(),
        "mean_lf_hf": df["lf_hf"].mean(),
        "n": len(df)
    }